# BetHero — full-budget training on Colab's free GPU

Trains all 8 model_keys (BTTS, Cards_3.5, Goals_1.5/2.5/3.5, Corners_7.5/9.5, 1x2) at `n_trials=20` using the local feature-cache snapshot — no live Supabase queries during training.

**One-time setup before running this notebook:**
1. Push the repo to a GitHub repo (private is fine) — Colab clones from there.
2. Upload `data_snapshot/` (131MB) to a folder in your Google Drive, e.g. `MyDrive/bethero/data_snapshot/`.
3. Add three Colab secrets (key icon in the left sidebar): `SUPABASE_URL`, `SUPABASE_SERVICE_ROLE_KEY`, and `GITHUB_TOKEN` (a fine-grained personal access token scoped to this repo, "Contents: Read-only" — needed because the repo is private and plain `git clone` has no way to log in). Never hardcode any of these in a cell.
4. Runtime → Change runtime type → **T4 GPU** (or whatever free GPU is offered).

Trained models are written straight to a Drive-mounted folder as each model_key finishes — a disconnect only loses the one market in flight, everything already done is safe. Re-running the notebook skips model_keys that already have a saved `StackingEnsemble_*.joblib` in that folder.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Repo is private, so cloning needs a GitHub token — add one as a Colab
# secret named GITHUB_TOKEN (key icon, left sidebar), not pasted here.
# Create it at https://github.com/settings/tokens (fine-grained token,
# scoped to just this repo, "Contents: Read-only" permission is enough).
from google.colab import userdata

GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
REPO_URL = f"https://{GITHUB_TOKEN}@github.com/joeie5/bethero.git"

!git clone $REPO_URL /content/bethero
%cd /content/bethero

In [ ]:
# Only what training actually needs — not the full app (FastAPI, Celery,
# Anthropic/Gemini clients, NewsAPI, etc. are irrelevant here). Mostly
# unpinned: requirements.txt's exact pins don't have prebuilt wheels for
# Colab's Python version, so pip was falling back to slow from-source
# builds (20+ minutes and still going) — confirmed live. Also skip torch
# entirely — Colab already ships a CUDA-enabled build.
# supabase IS pinned though — app/database.py builds a ClientOptions(...)
# with fields (storage_client_timeout) that a newer supabase-py version
# restructured, confirmed live: "'ClientOptions' object has no attribute
# 'storage'". supabase-py itself is pure Python, so pinning it doesn't
# reintroduce the slow-build problem the other packages had.
# sqlalchemy is required by app/database.py but isn't even listed in
# requirements.txt at all — a pre-existing gap, added here explicitly.
!pip install -q pandas numpy scikit-learn xgboost lightgbm joblib optuna mlflow "supabase==2.10.0" python-dotenv sqlalchemy pydantic pydantic-settings

In [ ]:
import os
from google.colab import userdata

os.environ["SUPABASE_URL"] = userdata.get("SUPABASE_URL")
os.environ["SUPABASE_SERVICE_ROLE_KEY"] = userdata.get("SUPABASE_SERVICE_ROLE_KEY")

# Separate folder per phase — Phase A (limit=1000) already saved
# StackingEnsemble_*.joblib for all 8 markets under models/, and the
# training loop's skip-logic checks by filename, so reusing that same
# folder for Phase B would skip everything thinking it's already done.
# Keeping Phase A's files untouched as a fallback until Phase B finishes.
PHASE_FOLDER = "models_phase_b"   # change to "models" for the final full run

DRIVE_MODEL_DIR = f"/content/drive/MyDrive/bethero/{PHASE_FOLDER}"
os.makedirs(DRIVE_MODEL_DIR, exist_ok=True)
os.environ["MODEL_DIR"] = DRIVE_MODEL_DIR

DATA_DIR = "/content/drive/MyDrive/bethero/data_snapshot"
assert os.path.isdir(DATA_DIR), f"Upload data_snapshot/ to {DATA_DIR} first"
print("MODEL_DIR ->", DRIVE_MODEL_DIR)
print("DATA_DIR  ->", DATA_DIR)

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only — check Runtime > Change runtime type")

In [ ]:
import sys
sys.path.insert(0, "/content/bethero")

from pathlib import Path
from app.ml.models import MODEL_DIR
from app.ml.training_pipeline_logic import train_market

MODEL_KEYS = [
    "BTTS", "Cards_3.5",
    "Goals_1.5", "Goals_2.5", "Goals_3.5",
    "Corners_7.5", "Corners_9.5",
    "1x2",
]
N_TRIALS = 20
N_META_TRIALS = 20

# Phase B — real, usable-quality models. 50,000 rows against 128 features
# gives ~35,000 train rows after the temporal split — a features:rows
# ratio of about 1:273, healthy enough for the model to learn genuine
# patterns instead of memorizing noise (vs Phase A's 1:5.5 at limit=1000,
# which only proved the pipeline mechanics work, not model quality).
# Estimated ~1 hour per market, ~8 hours total across all 8 — a real step
# down from the full ~290k-row run (~25-40h) while still being usable.
LIMIT = 50000

print("MODEL_DIR resolved to:", MODEL_DIR)
assert str(MODEL_DIR) == os.environ["MODEL_DIR"], "MODEL_DIR env var must be set BEFORE importing app.ml.* — restart runtime and re-run from the top"

In [ ]:
import time

for model_key in MODEL_KEYS:
    ensemble_path = MODEL_DIR / f"StackingEnsemble_football_{model_key}.joblib"
    if ensemble_path.exists():
        print(f"=== {model_key}: already trained, skipping ({ensemble_path}) ===")
        continue

    print(f"\n=== {model_key} — starting (limit={LIMIT}) ===", flush=True)
    t0 = time.monotonic()
    results = train_market(
        "football", model_key,
        n_trials=N_TRIALS, n_meta_trials=N_META_TRIALS,
        limit=LIMIT,
        data_dir=Path(DATA_DIR),
    )
    elapsed = time.monotonic() - t0
    for r in results:
        print(f"  {r.model_name}: {r.status} {r.error or ''} ({r.duration_s}s)")
        if r.calibration_table is not None:
            print(r.calibration_table.to_string(index=False))
    print(f"{model_key} done in {elapsed/60:.1f} min", flush=True)

print("\n=== ALL MARKETS DONE ===")

## After this finishes
Trained model files are already sitting in your Drive at `MyDrive/bethero/models/` — copy that folder over your local `models/` directory (or point `MODEL_DIR` at it directly) to make them available to `Predictor`.